# Data preparation (CPU session — free, no GPU quota)

1. Download HyperKvasir unlabeled (~24 GB) into `/kaggle/tmp` scratch and stream-resize to 256px (~3 GB), then publish as a private Kaggle Dataset.
2. Perceptual-hash the corpus against Kvasir-SEG and exclude near-duplicates — this prevents pretraining/test contamination and the count goes in the thesis.
3. Generate the group-aware, stratified Kvasir-SEG splits.

**Set the accelerator to None.** This is pure CPU work and a GPU session would burn quota for nothing.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command, streaming its output live.

    Streaming rather than capture_output matters because these jobs run for
    tens of minutes and report progress as they go. Buffering that until the
    process exits makes a long job indistinguishable from a hung one.
    """
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: a child process writing to a pipe switches from line
    # buffering to 4 KB block buffering, so progress lines would still arrive
    # in bursts (or not at all until exit) even though we stream them here.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code_ = p.wait()
    if check and code_ != 0:
        # Include the tail of the output in the exception. Otherwise the
        # traceback shows only this wrapper and the real error is buried
        # further up the cell, which is easy to miss and impossible to
        # copy/paste usefully.
        tail = ''.join(lines[-25:]).rstrip()
        raise RuntimeError(
            f'command failed (exit {code_}): {cmd}\n\n--- last output ---\n{tail}')
    return subprocess.CompletedProcess(cmd, code_, ''.join(lines), '')

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# Credentials for publishing the resized corpora as Kaggle Datasets.
# Labels must be exactly KAGGLE_USERNAME and KAGGLE_KEY (Add-ons -> Secrets).
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')
print('credentials loaded for', os.environ['KAGGLE_USERNAME'])


In [ ]:
# Locate the mounted HyperKvasir source. Reading the attached dataset beats
# downloading from datasets.simula.no, whose TLS chain is missing an
# intermediate certificate (curl exit 60) — and a mount costs no disk quota.
import glob

# Kaggle nests mounts differently over time (it used to be
# /kaggle/input/<slug>/, now it can be /kaggle/input/datasets/<owner>/...),
# so search by directory NAME at any depth rather than by a fixed path.
ROOT = '/kaggle/input'
MAX_DEPTH = 8

def walk_bounded(root=ROOT, max_depth=MAX_DEPTH):
    base = root.rstrip('/').count(os.sep)
    for cur, dirs, files in os.walk(root):
        if cur.count(os.sep) - base >= max_depth:
            dirs[:] = []
        yield cur, dirs, files

def find_dir(*names):
    """Shallowest directory whose basename matches any of `names`."""
    hits = [cur for cur, _, _ in walk_bounded()
            if os.path.basename(cur) in names]
    return sorted(hits, key=lambda p: (p.count(os.sep), p))[0] if hits else None

def show_tree(limit=60):
    out = []
    for cur, _, files in walk_bounded(max_depth=4):
        n = sum(1 for f in files if f.lower().endswith(('.jpg','.jpeg','.png')))
        out.append(f'{cur}   ({n} images)' if n else cur)
        if len(out) >= limit:
            out.append('... (truncated)')
            break
    return '\n  '.join(out)

print('=== /kaggle/input tree ===')
print(' ', show_tree())

UNLABELED_SRC = find_dir('unlabeled-images')
LABELED_SRC = find_dir('labeled-images')

# The unlabeled split has its JPEGs one level down, in images/.
if UNLABELED_SRC and os.path.isdir(os.path.join(UNLABELED_SRC, 'images')):
    UNLABELED_SRC = os.path.join(UNLABELED_SRC, 'images')

missing = []
if not UNLABELED_SRC:
    missing.append('hyper-kvasir-unlabeled-images  (faisalmahmud69, 29.4 GB) '
                   '- the pretraining corpus, ~99,417 images')
if not LABELED_SRC:
    missing.append('hyper-kvasir-labeled-images    (faustkkk or faisalmahmud69, 3.9 GB) '
                   '- only used by the k-NN probe')
if missing:
    found = [f'{k} = {v}' for k, v in
             [('unlabeled', UNLABELED_SRC), ('labeled', LABELED_SRC)] if v]
    raise FileNotFoundError(
        'Still need to attach:\n  - ' + '\n  - '.join(missing) +
        ('\n\nAlready found:\n  ' + '\n  '.join(found) if found else '') +
        '\n\nFix: + Add Input -> Datasets -> search the name above -> Add. '
        'Large datasets take a few minutes to mount; wait for the sidebar '
        'entry before re-running this cell.'
        '\n\nTree:\n  ' + show_tree())

print('unlabeled:', UNLABELED_SRC, '->', len(os.listdir(UNLABELED_SRC)), 'entries')
print('labeled  :', LABELED_SRC, '->', sorted(os.listdir(LABELED_SRC))[:5])


### Corpus size

`LIMIT = 0` builds the full ~99,417-image corpus (20-40 min) — that is what the thesis results need.

Set `LIMIT = 3000` for a fast end-to-end check of the whole pipeline (~1-2 min here, then ~6 min to pretrain, then segmentation). A subset publishes to its **own dataset slug**, so it can never be confused with the real corpus, and the pretraining log prints the count it actually loaded.

In [ ]:
LIMIT = 3000   # 0 = full corpus

SUFFIX = f'-sub{LIMIT}' if LIMIT else ''
UNLABELED_SLUG = 'morsalin101/hyperkvasir-unlabeled-256' + SUFFIX
LABELED_SLUG = 'morsalin101/hyperkvasir-labeled-256'
print('publishing corpus to:', UNLABELED_SLUG)
if LIMIT:
    print(f'*** SUBSET: {LIMIT} images. Pipeline validation only. ***')


In [ ]:
# The resize is what lets the dataloader keep up with the GPU later:
# decoding 1280x1024 JPEGs at 400 views/s is not possible on 4 cores.
#
# check=False: the resize is the expensive part and its output lands in
# /kaggle/working, which becomes this kernel's output when you Save Version.
# So a failed *publish* must not abort the notebook — the corpus is still
# usable via kernel_sources even if the Dataset upload does not go through.
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
r = sh(f'python scripts/build_hyperkvasir_dataset.py --source-dir {UNLABELED_SRC} {limit_arg} --publish {UNLABELED_SLUG}', check=False)

import pathlib
n_built = len(list(pathlib.Path('/kaggle/working/hk256').glob('*.jpg')))
if n_built == 0:
    raise RuntimeError('resize produced nothing — see the output above')
print(f'\ncorpus on disk: {n_built} images in /kaggle/working/hk256')
if r.returncode != 0:
    print('*** The Dataset upload failed, but the corpus itself is fine.\n'
          '    Continue running the remaining cells, then Save Version —\n'
          '    /kaggle/working becomes this kernel output and pretraining\n'
          '    can mount it via kernel_sources instead. ***')


In [ ]:
# The labelled split (10,662 images, 23 classes) — ~400 MB, a few minutes.
# Never trained on; used only by the k-NN / linear probe in the analysis
# notebook, which is the cheapest check that pretraining actually worked.
# Same non-fatal policy as the corpus cell above.
sh(f'python scripts/build_hyperkvasir_dataset.py --split labeled --source-dir {LABELED_SRC} --publish {LABELED_SLUG}', check=False)


In [ ]:
# Point at the corpus we just built in /kaggle/working — the published dataset
# only appears under /kaggle/input on a *later* session, so auto-resolution
# would not find it yet in this one.
sh('python scripts/dedup_phash.py --threshold 6 --pretrain-root /kaggle/working/hk256')


In [ ]:
sh('python -m src.data.splits')


In [ ]:
print(open('splits/dedup_report.json').read())
print(open('splits/800_100_100/stats.json').read())


### Copy `splits/` back into git — required before segmentation

The later notebooks clone the repo from GitHub, so the split files and the dedup exclusion list have to be **committed**, not just present here. The cell below copies them to `/kaggle/working/splits_to_commit/`; download that from the notebook's Output tab (or `kaggle kernels output`), drop it into `splits/` locally, and push.

This is what guarantees every run — yours and anyone reproducing it — uses byte-identical splits.

In [ ]:
import shutil
shutil.copytree('splits', '/kaggle/working/splits_to_commit', dirs_exist_ok=True)
for root, _, files in os.walk('/kaggle/working/splits_to_commit'):
    for f in sorted(files):
        print(os.path.join(root, f))
print('\nRetrieve with:\n'
      '  kaggle kernels output morsalin101/data-prep -p /tmp/dp\n'
      '  cp -r /tmp/dp/splits_to_commit/* splits/\n'
      '  git add splits && git commit -m \'data: splits + dedup list\' && git push')
